In [1]:
# Generate the report for the people who code as a hobby based on their gender and continent.

import pandas as pd
import numpy as np
import country_converter as coco

# Load dataset
df = pd.read_csv('../data/survey_results_public.csv', low_memory=False)
df = df.dropna()

# Normalize Gender values
def normalize_gender(val):
    if pd.isna(val):
        return 'Others'
    s = str(val).strip()
    if s in ['Man', 'Woman']:
        return s
    else:
        return 'Others'

df['Gender'] = df['Gender'].apply(normalize_gender)

#  Country to Continent mapping
cc = coco.CountryConverter()
df = df[df['Country'] != 'Other Country (Not Listed Above)']
df['Continent'] = cc.convert(names=df['Country'], to='continent', not_found='Unknown')

# Create HobbyistFlag
df['HobbyistFlag'] = df['Hobbyist'].map(lambda x: True if str(x).strip() == 'Yes' else False)

# Group by Continent and Gender
result = df.groupby(['Continent','Gender']).agg(
    Total_respondents=('Respondent','nunique'),
    Hobbyists=('HobbyistFlag','sum')
).reset_index()

# Calculate percentage of hobbyists
result['pct_hobbyist'] = (result['Hobbyists'] / result['Total_respondents'] * 100).round(2)

# Save CSV
import os
os.makedirs('../output', exist_ok=True)
result.to_csv('../output/q5_code_as_hobby.csv', index=False)

# Display the first 10 rows of the result
result.head(10)

,Continent,Gender,Total_respondents,Hobbyists,pct_hobbyist
0,Africa,Man,65,56,86.15
1,Africa,Woman,4,2,50.00
2,America,Man,1399,1210,86.49
3,America,Others,11,11,100.00
4,America,Woman,58,40,68.97
5,Asia,Man,516,428,82.95
6,Asia,Others,2,2,100.00
7,Asia,Woman,13,7,53.85
8,Europe,Man,1246,1093,87.72
9,Europe,Others,1,1,100.00
